In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,AutoModelForMaskedLM
from datasets import Dataset
import torch

In [27]:
df=pd.read_csv(r'C:\Users\hp\chatbot_health\backend\datasets\intent\intent_final2.csv')

In [28]:
labels = df["intent"].unique().tolist()
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["intent"].map(label2id)
print(label2id)
print(id2label)


{'demande_consultation': 0, 'demande_urgence': 1, 'description_symptome': 2, 'demande_posologie': 3, 'remboursement_mutuelle': 4, 'demande_medicament': 5, 'demande_prix': 6}
{0: 'demande_consultation', 1: 'demande_urgence', 2: 'description_symptome', 3: 'demande_posologie', 4: 'remboursement_mutuelle', 5: 'demande_medicament', 6: 'demande_prix'}


In [29]:
train, temp = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

In [30]:
print(f"Train size: {len(train)}, Validation size: {len(val)}, Test size: {len(test)}")

Train size: 852, Validation size: 183, Test size: 183


In [31]:
MODEL_PATH = "../models/intent_model/"
TOKENIZER_PATH = "../models/intent_model/tokenizer/"

In [32]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

In [33]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train[["text", "label"]])
val_dataset = Dataset.from_pandas(val[["text", "label"]])
test_dataset = Dataset.from_pandas(test[["text", "label"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 183/183 [00:00<00:00, 5367.65 examples/s]


In [34]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2356.03it/s]


In [35]:
import accelerate
print(accelerate.__version__)

1.13.0


In [36]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no", 
    load_best_model_at_end=False,
    logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [38]:
trainer.train()

c:\Users\hp\chatbot_health\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.709134
2,No log,0.512297
3,No log,0.442977
4,No log,0.434106
5,No log,0.357388


TrainOutput(global_step=270, training_loss=0.4328531618471499, metrics={'train_runtime': 1760.1372, 'train_samples_per_second': 2.42, 'train_steps_per_second': 0.153, 'total_flos': 52542347544000.0, 'train_loss': 0.4328531618471499, 'epoch': 5.0})

In [48]:

model.save_pretrained("../models/intent_model_v2/")
tokenizer.save_pretrained("../models/intent_model_v2/tokenizer/")

Writing model shards: 100%|██████████| 1/1 [00:34<00:00, 34.46s/it]


('../models/intent_model_v2/tokenizer/tokenizer_config.json',
 '../models/intent_model_v2/tokenizer/tokenizer.json')

In [42]:
predictions = trainer.predict(test_dataset)
print(predictions.metrics)

{'test_loss': 0.3712829053401947, 'test_runtime': 22.2074, 'test_samples_per_second': 8.24, 'test_steps_per_second': 0.54}


In [47]:
import numpy as np
from sklearn.metrics import classification_report

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=list(label2id.keys())))

                        precision    recall  f1-score   support

  demande_consultation       0.89      1.00      0.94        24
       demande_urgence       0.88      0.84      0.86        25
  description_symptome       0.82      0.88      0.85        26
     demande_posologie       0.85      0.85      0.85        26
remboursement_mutuelle       0.97      1.00      0.98        28
    demande_medicament       0.93      0.96      0.94        26
          demande_prix       1.00      0.79      0.88        28

              accuracy                           0.90       183
             macro avg       0.90      0.90      0.90       183
          weighted avg       0.91      0.90      0.90       183



In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

phrases = [
    
   ""
    
      
    

]

for phrase in phrases:
    result = classifier(phrase)
    print(f"{phrase} → {result[0]['label']} ({result[0]['score']:.2f})")

i want to know with dwa to take   → demande_urgence (0.98)
